# 04 · 离线评测没有开始（HANDOFF §6 步骤 4）

**对应 HANDOFF §6 步骤 4**「离线在 C & D 上评，填 TRAINING_PLAN 里的对照表」。

**前置条件已经完成**，评测本身没有开始——因为它需要 P2 训出来的检查点，
而 P2 一个 run 都没跑（见 [03](03_p2_training_matrix.ipynb)）。


In [1]:
%matplotlib inline
import json, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "font.size": 9,
    "axes.grid": True, "grid.alpha": .25,
    "axes.spines.top": False, "axes.spines.right": False,
})
R = Path("/lus/lfs1aip2/projects/public/u6gb/tasks/large-discovery-model/ldm_rl/results")
def load(name): return json.loads((R / name).read_text())

import os
base = Path("/lus/lfs1aip2/projects/public/u6gb/tasks/large-discovery-model/ldm_rl/results")
rows = []
for name, sub in [("G12C QSAR 模型", "g12c_qsar_20260901T010923Z"),
                  ("G12D QSAR（配对）", "g12d_qsar_matched_20260901T011826Z")]:
    p = base / sub / "best_model.joblib"
    rows.append([name, "已训好" if p.exists() else "缺",
                 f"{p.stat().st_size/1024:.0f} KB" if p.exists() else "-", str(p)])
pd.DataFrame(rows, columns=["前置产物", "状态", "大小", "路径"]).style.hide(axis="index")

前置产物,状态,大小,路径
G12C QSAR 模型,已训好,3245 KB,/lus/lfs1aip2/projects/public/u6gb/tasks/large-discovery-model/ldm_rl/results/g12c_qsar_20260901T010923Z/best_model.joblib
G12D QSAR（配对）,已训好,1051 KB,/lus/lfs1aip2/projects/public/u6gb/tasks/large-discovery-model/ldm_rl/results/g12d_qsar_matched_20260901T011826Z/best_model.joblib


**读法**：HANDOFF 里标为「可并行的前置」的那一项
（`train_g12c_qsar.py` 训出 `best_g12c_model.joblib`）**已经完成**，
两个 QSAR 模型都在。G12D 活性模型本来就随 repo 提供。

所以评测缺的**只是被评的对象**。

## 评测要产出什么

TRAINING_PLAN 的对照表需要每个 run 在 **C（G12C）** 与 **D（G12D）**
两个靶点上的离线打分。注意 HANDOFF §开头写明的方向：

> **train KRAS G12D → eval G12C & G12D**（G12C 只在离线评测用，不进训练循环）

也就是说 G12C 是**留出的泛化检验**——训练时模型从没见过它。
这一点决定了评测不能用训练时那套共享 GP：
那个 GP 里全是 G12D 的评测结果。

## 能先做的事

即使没有检查点，有两件不依赖 P2 的准备可以先落地：

1. **把评测脚本跑通一遍**，用 base 模型或 SFT 模型当占位输入，
   确认打分链路（docking → activity → 汇总）在 aarch64 上没有新坑。
   这与 §6 P0 的思路一致：先验管道，再验结论。
2. **确定对照表的口径**：每个 run 报几个数、用哪些种子、
   非劣/提升怎么判。**这要在看到结果之前定下来**，否则任何结果
   都能被解释成成功。